# Build Dataset — M5 V4 (Full History, All Events)

Builds a clean 7-day aggregated dataset from the **full M5 dataset** (2011–2016).

- **Store:** CA_1 only
- **Departments:** FOODS_1, FOODS_2, FOODS_3
- **Items:** ~1,437 SKUs
- **History:** ~5 years (Jan 2011 → Apr 2016)
- **Split:** 70 / 15 / 15 (train / val / test)
- **Events:** All 28 M5 calendar events (13 original + 15 newly added US & cultural)
- Days with two simultaneous events → both flags set to 1

**Output: 35 columns** — item_id, date, target, 3 numeric features, 29 event flags

## 1) Imports & Paths

In [40]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
M5   = ROOT / 'data' / 'raw' / 'm5'
OUT  = ROOT / 'data' / 'processed' / '7_Day_Dataset'
OUT.mkdir(parents=True, exist_ok=True)

print('M5 raw dir :', M5)
print('Output dir :', OUT)

M5 raw dir : C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\raw\m5
Output dir : C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\processed\7_Day_Dataset


## 2) Load Raw Files

In [41]:
sales_df    = pd.read_csv(M5 / 'sales_train_validation.csv')
calendar_df = pd.read_csv(M5 / 'calendar.csv', parse_dates=['date'])
prices_df   = pd.read_csv(M5 / 'sell_prices.csv')

print(f'Sales shape    : {sales_df.shape}')
print(f'Calendar shape : {calendar_df.shape}')
print(f'Prices shape   : {prices_df.shape}')
print(f'Date range     : {calendar_df["date"].min().date()} → {calendar_df["date"].max().date()}')

Sales shape    : (30490, 1919)
Calendar shape : (1969, 14)
Prices shape   : (6841121, 4)
Date range     : 2011-01-29 → 2016-06-19


## 3) Filter to CA_1 / FOODS_1/2/3

In [42]:
FOOD_DEPTS = ['FOODS_1', 'FOODS_2', 'FOODS_3']

mask     = (sales_df['store_id'] == 'CA_1') & (sales_df['dept_id'].isin(FOOD_DEPTS))
sales_df = sales_df[mask].reset_index(drop=True)

print(f'Items after filter : {len(sales_df)}')
print(sales_df['dept_id'].value_counts().to_dict())

Items after filter : 1437
{'FOODS_3': 823, 'FOODS_2': 398, 'FOODS_1': 216}


## 4) Melt Wide → Long (daily rows)

In [43]:
id_cols  = ['item_id', 'store_id', 'dept_id']
day_cols = [c for c in sales_df.columns if c.startswith('d_')]

long_df = sales_df[id_cols + day_cols].melt(
    id_vars=id_cols, var_name='d', value_name='sales'
)

print(f'Long format shape: {long_df.shape}')
print(long_df.head(3))

Long format shape: (2748981, 5)
       item_id store_id  dept_id    d  sales
0  FOODS_1_001     CA_1  FOODS_1  d_1      3
1  FOODS_1_002     CA_1  FOODS_1  d_1      0
2  FOODS_1_003     CA_1  FOODS_1  d_1      0


## 5) Merge Calendar (date, events, snap_CA)

In [44]:
calendar_slim = calendar_df[['d', 'date', 'wm_yr_wk', 'snap_CA',
                              'event_name_1', 'event_name_2']].copy()

long_df = long_df.merge(calendar_slim, on='d', how='left')
print(f'After calendar merge: {long_df.shape}')
print(f'Date range: {long_df["date"].min().date()} → {long_df["date"].max().date()}')

After calendar merge: (2748981, 10)
Date range: 2011-01-29 → 2016-04-24


## 6) Merge Sell Prices

In [45]:
prices_slim = prices_df[prices_df['store_id'] == 'CA_1'][['item_id', 'wm_yr_wk', 'sell_price']]

long_df = long_df.merge(prices_slim, on=['item_id', 'wm_yr_wk'], how='left')

# ffill: hold last known price forward (price unchanged until next week's entry)
# bfill: fill early nulls before the item's first recorded price using its first price
long_df = long_df.sort_values(['item_id', 'date'])
long_df['sell_price'] = (
    long_df.groupby('item_id')['sell_price']
    .transform(lambda s: s.ffill().bfill())
)

# Fallback: any item with absolutely no price → global median (edge case)
global_median = long_df['sell_price'].median()
long_df['sell_price'] = long_df['sell_price'].fillna(global_median)

null_price = long_df['sell_price'].isna().sum()
print(f'Null sell_price rows after fill: {null_price}')
print(f'After price merge: {long_df.shape}')

Null sell_price rows after fill: 0
After price merge: (2748981, 11)


## 7) Build Event Binary Columns + is_month_end

In [46]:
# Exact event_name strings as they appear in M5 calendar.csv
# Both event_name_1 and event_name_2 are checked — days with two events get both flags = 1
EVENT_MAP = {
    # --- Previously included ---
    'event_christmas_7':          'Christmas',
    'event_easter_7':             'Easter',
    'event_eid_al_fitr_7':        'Eid al-Fitr',
    'event_eid_al_adha_7':        'EidAlAdha',
    'event_fathers_day_7':        "Father's day",
    'event_halloween_7':          'Halloween',
    'event_mothers_day_7':        "Mother's day",
    'event_newyear_7':            'NewYear',
    'event_orthodox_christmas_7': 'OrthodoxChristmas',
    'event_orthodox_easter_7':    'OrthodoxEaster',
    'event_ramadan_starts_7':     'Ramadan starts',
    'event_thanksgiving_7':       'Thanksgiving',
    'event_valentines_day_7':     'ValentinesDay',
    # --- US national & cultural events (newly added) ---
    'event_superbowl_7':          'SuperBowl',
    'event_independence_day_7':   'IndependenceDay',
    'event_memorial_day_7':       'MemorialDay',
    'event_labor_day_7':          'LaborDay',
    'event_mlk_day_7':            'MartinLutherKingDay',
    'event_presidents_day_7':     'PresidentsDay',
    'event_columbus_day_7':       'ColumbusDay',
    'event_veterans_day_7':       'VeteransDay',
    'event_st_patricks_day_7':    'StPatricksDay',
    'event_cinco_de_mayo_7':      'Cinco De Mayo',
    'event_chanukah_7':           'Chanukah End',
    'event_lent_start_7':         'LentStart',
    'event_lent_week2_7':         'LentWeek2',
    'event_pesach_end_7':         'Pesach End',
    'event_purim_end_7':          'Purim End',
}

for col, name in EVENT_MAP.items():
    long_df[col] = (
        (long_df['event_name_1'] == name) | (long_df['event_name_2'] == name)
    ).astype(int)

long_df['is_month_end'] = long_df['date'].dt.is_month_end.astype(int)

# Sanity check event coverage
print(f'Total event columns: {len(EVENT_MAP)}')
print('Event days per column:')
for col in EVENT_MAP:
    print(f'  {col}: {long_df[col].sum()} rows')

Total event columns: 28
Event days per column:
  event_christmas_7: 7185 rows
  event_easter_7: 8622 rows
  event_eid_al_fitr_7: 7185 rows
  event_eid_al_adha_7: 7185 rows
  event_fathers_day_7: 7185 rows
  event_halloween_7: 7185 rows
  event_mothers_day_7: 7185 rows
  event_newyear_7: 7185 rows
  event_orthodox_christmas_7: 7185 rows
  event_orthodox_easter_7: 7185 rows
  event_ramadan_starts_7: 7185 rows
  event_thanksgiving_7: 7185 rows
  event_valentines_day_7: 8622 rows
  event_superbowl_7: 8622 rows
  event_independence_day_7: 7185 rows
  event_memorial_day_7: 7185 rows
  event_labor_day_7: 7185 rows
  event_mlk_day_7: 7185 rows
  event_presidents_day_7: 8622 rows
  event_columbus_day_7: 7185 rows
  event_veterans_day_7: 7185 rows
  event_st_patricks_day_7: 8622 rows
  event_cinco_de_mayo_7: 7185 rows
  event_chanukah_7: 7185 rows
  event_lent_start_7: 8622 rows
  event_lent_week2_7: 8622 rows
  event_pesach_end_7: 7185 rows
  event_purim_end_7: 8622 rows


## 8) Assign 7-Day Window Index per Item

In [47]:
long_df = long_df.sort_values(['item_id', 'date']).reset_index(drop=True)
long_df['week_idx'] = long_df.groupby('item_id').cumcount() // 7

print(f'Unique weeks per item (sample): {long_df.groupby("item_id")["week_idx"].max().describe()}')

Unique weeks per item (sample): count    1437.0
mean      273.0
std         0.0
min       273.0
25%       273.0
50%       273.0
75%       273.0
max       273.0
Name: week_idx, dtype: float64


## 9) Aggregate to Weekly Rows

In [48]:
event_cols = list(EVENT_MAP.keys())

agg_dict = {
    'date':                  ('date',        'first'),
    'aggregated_sales_7':    ('sales',       'sum'),
    'aggregated_sell_price': ('sell_price',  'mean'),
    'is_month_end':          ('is_month_end','max'),
    'snap_ca':               ('snap_CA',     'max'),
}
for col in event_cols:
    agg_dict[col] = (col, 'max')

agg = (
    long_df.groupby(['item_id', 'week_idx'])
    .agg(**agg_dict)
    .reset_index()
    .drop(columns='week_idx')
)

print(f'Weekly dataset shape: {agg.shape}')
print(f'Columns: {list(agg.columns)}')
print(f'Date range: {agg["date"].min().date()} → {agg["date"].max().date()}')

Weekly dataset shape: (393738, 34)
Columns: ['item_id', 'date', 'aggregated_sales_7', 'aggregated_sell_price', 'is_month_end', 'snap_ca', 'event_christmas_7', 'event_easter_7', 'event_eid_al_fitr_7', 'event_eid_al_adha_7', 'event_fathers_day_7', 'event_halloween_7', 'event_mothers_day_7', 'event_newyear_7', 'event_orthodox_christmas_7', 'event_orthodox_easter_7', 'event_ramadan_starts_7', 'event_thanksgiving_7', 'event_valentines_day_7', 'event_superbowl_7', 'event_independence_day_7', 'event_memorial_day_7', 'event_labor_day_7', 'event_mlk_day_7', 'event_presidents_day_7', 'event_columbus_day_7', 'event_veterans_day_7', 'event_st_patricks_day_7', 'event_cinco_de_mayo_7', 'event_chanukah_7', 'event_lent_start_7', 'event_lent_week2_7', 'event_pesach_end_7', 'event_purim_end_7']
Date range: 2011-01-29 → 2016-04-23


## 11) Enforce Column Order

In [49]:
FINAL_COLS = [
    'item_id', 'date', 'aggregated_sales_7',
    'is_month_end', 'aggregated_sell_price', 'snap_ca',
    # original events
    'event_christmas_7', 'event_easter_7', 'event_eid_al_fitr_7',
    'event_eid_al_adha_7', 'event_fathers_day_7', 'event_halloween_7',
    'event_mothers_day_7', 'event_newyear_7', 'event_orthodox_christmas_7',
    'event_orthodox_easter_7', 'event_ramadan_starts_7',
    'event_thanksgiving_7', 'event_valentines_day_7',
    # newly added US & cultural events
    'event_superbowl_7', 'event_independence_day_7', 'event_memorial_day_7',
    'event_labor_day_7', 'event_mlk_day_7', 'event_presidents_day_7',
    'event_columbus_day_7', 'event_veterans_day_7', 'event_st_patricks_day_7',
    'event_cinco_de_mayo_7', 'event_chanukah_7', 'event_lent_start_7',
    'event_lent_week2_7', 'event_pesach_end_7', 'event_purim_end_7',
]

agg = agg[FINAL_COLS].copy()
print(f'Final column count : {len(agg.columns)}')
print(f'Columns            : {list(agg.columns)}')
print(f'Null values        : {agg.isna().sum().sum()}')

Final column count : 34
Columns            : ['item_id', 'date', 'aggregated_sales_7', 'is_month_end', 'aggregated_sell_price', 'snap_ca', 'event_christmas_7', 'event_easter_7', 'event_eid_al_fitr_7', 'event_eid_al_adha_7', 'event_fathers_day_7', 'event_halloween_7', 'event_mothers_day_7', 'event_newyear_7', 'event_orthodox_christmas_7', 'event_orthodox_easter_7', 'event_ramadan_starts_7', 'event_thanksgiving_7', 'event_valentines_day_7', 'event_superbowl_7', 'event_independence_day_7', 'event_memorial_day_7', 'event_labor_day_7', 'event_mlk_day_7', 'event_presidents_day_7', 'event_columbus_day_7', 'event_veterans_day_7', 'event_st_patricks_day_7', 'event_cinco_de_mayo_7', 'event_chanukah_7', 'event_lent_start_7', 'event_lent_week2_7', 'event_pesach_end_7', 'event_purim_end_7']
Null values        : 0


## 12) 70 / 15 / 15 Time-Based Split

In [50]:
dates = sorted(agg['date'].unique())
n     = len(dates)

train_dates = dates[:int(0.70 * n)]
val_dates   = dates[int(0.70 * n):int(0.85 * n)]
test_dates  = dates[int(0.85 * n):]

train_df = agg[agg['date'].isin(train_dates)].reset_index(drop=True)
val_df   = agg[agg['date'].isin(val_dates)].reset_index(drop=True)
test_df  = agg[agg['date'].isin(test_dates)].reset_index(drop=True)

print(f'Total unique dates : {n}')
print(f'Train  : {len(train_dates)} weeks | {len(train_df):>7,} rows | {pd.Timestamp(train_dates[0]).date()} → {pd.Timestamp(train_dates[-1]).date()}')
print(f'Val    : {len(val_dates)} weeks | {len(val_df):>7,} rows | {pd.Timestamp(val_dates[0]).date()} → {pd.Timestamp(val_dates[-1]).date()}')
print(f'Test   : {len(test_dates)} weeks | {len(test_df):>7,} rows | {pd.Timestamp(test_dates[0]).date()} → {pd.Timestamp(test_dates[-1]).date()}')
print(f'Total  : {len(train_df) + len(val_df) + len(test_df):>7,} rows')

Total unique dates : 274
Train  : 191 weeks | 274,467 rows | 2011-01-29 → 2014-09-20
Val    : 41 weeks |  58,917 rows | 2014-09-27 → 2015-07-04
Test   : 42 weeks |  60,354 rows | 2015-07-11 → 2016-04-23
Total  : 393,738 rows


## 13) Sanity Checks

In [51]:
# 1. No date overlap between splits
assert len(set(train_dates) & set(val_dates)) == 0, 'OVERLAP: train/val'
assert len(set(val_dates)   & set(test_dates)) == 0, 'OVERLAP: val/test'
print('No date overlap between splits')

# 2. All items present in train
train_items = set(train_df['item_id'])
all_items   = set(agg['item_id'])
assert train_items == all_items, f'Missing items in train: {all_items - train_items}'
print(f'All {len(all_items)} items present in train')

# 3. snap_ca binary
assert agg['snap_ca'].isin([0, 1]).all(), 'snap_ca has non-binary values'
print(f'snap_ca is binary — CA_1 SNAP weeks: {int(agg["snap_ca"].sum())} / {len(agg)}')

# 4. No nulls in final dataset
assert agg.isna().sum().sum() == 0, 'Null values found!'
print('No null values')

# 5. Target summary
print(f'\naggregated_sales_7 summary:')
print(agg['aggregated_sales_7'].describe().round(2))

No date overlap between splits
All 1437 items present in train
snap_ca is binary — CA_1 SNAP weeks: 206928 / 393738
No null values

aggregated_sales_7 summary:
count    393738.00
mean         13.67
std          33.08
min           0.00
25%           0.00
50%           5.00
75%          14.00
max        2134.00
Name: aggregated_sales_7, dtype: float64


## 14) Save to Disk

In [52]:
train_df.to_csv(OUT / 'train.csv', index=False)
val_df.to_csv(OUT   / 'val.csv',   index=False)
test_df.to_csv(OUT  / 'test.csv',  index=False)

print(f'Saved to {OUT}')
print(f'  train.csv : {len(train_df):,} rows')
print(f'  val.csv   : {len(val_df):,} rows')
print(f'  test.csv  : {len(test_df):,} rows')

Saved to C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\processed\7_Day_Dataset
  train.csv : 274,467 rows
  val.csv   : 58,917 rows
  test.csv  : 60,354 rows
